# From Voice to Vision — 2. Preprocessing and Feature Extraction

Each clip is resampled to 16 kHz, denoised by spectral gating, trimmed of leading and trailing
silence and padded or centre-cropped to a fixed duration of three seconds, so that every
downstream representation has a constant shape.

Two complementary feature families are then extracted: a 128-band **log-Mel spectrogram** for the
convolutional model, and a 160-dimensional **MFCC summary vector** (mean and standard deviation of
the MFCCs and of their first delta) for the classical baselines. The extracted arrays are cached
on disk so that subsequent notebooks load them instantly.

In [ ]:
# Clone the project repository and install the audio dependencies
REPO_URL = "https://github.com/Nadaa3672/from-voice-to-vision.git"
import os
repo = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
if not os.path.exists(repo):
    !git clone $REPO_URL
%cd $repo
!git pull -q
!pip install -q librosa soundfile noisereduce tqdm

In [ ]:
from src import config, data_loader, preprocessing, features
data_loader.download_ravdess()
df = data_loader.build_index()
print("Clips:", len(df))

## Effect of denoising

Spectral gating attenuates stationary background noise. RAVDESS is studio-recorded and therefore
already clean, so the effect is subtle: the spectrogram background darkens while the harmonic
structure of the voice is preserved.

In [ ]:
import librosa, librosa.display, numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

row = df[df.emotion == "happy"].iloc[0]
raw = preprocessing.load_audio(row.path)
den = preprocessing.reduce_noise(raw)
proc = preprocessing.preprocess(row.path, denoise=True)

print(f"Raw duration: {len(raw)/config.SAMPLE_RATE:.2f} s | "
      f"after fixed-length padding: {len(proc)/config.SAMPLE_RATE:.2f} s")
display(Audio(raw, rate=config.SAMPLE_RATE)); display(Audio(den, rate=config.SAMPLE_RATE))

fig, ax = plt.subplots(2, 2, figsize=(13, 7))
for j, (sig, ttl) in enumerate([(raw, "Raw"), (den, "Denoised")]):
    librosa.display.waveshow(sig, sr=config.SAMPLE_RATE, ax=ax[0, j])
    ax[0, j].set_title(f"Waveform — {ttl}")
    ax[0, j].set_xlabel("Time (s)"); ax[0, j].set_ylabel("Amplitude")
    S = librosa.power_to_db(librosa.feature.melspectrogram(
        y=sig, sr=config.SAMPLE_RATE, n_mels=config.N_MELS,
        n_fft=config.N_FFT, hop_length=config.HOP_LENGTH), ref=np.max)
    librosa.display.specshow(S, sr=config.SAMPLE_RATE, hop_length=config.HOP_LENGTH,
                             x_axis="time", y_axis="mel", ax=ax[1, j])
    ax[1, j].set_title(f"log-Mel spectrogram — {ttl}"); ax[1, j].set_xlabel("Time (s)")
plt.tight_layout()
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(config.FIGURES_DIR / "01_preprocessing.png", dpi=150)
plt.show()

## Feature extraction over the whole corpus

The main dataset is built without denoising: RAVDESS is already clean, and spectral gating risks
removing low-energy cues that carry emotional information. The denoised variant remains available
as a controlled comparison.

In [ ]:
data = features.build_dataset(df, denoise=False, cache=True)
print("log-Mel tensor:", data["X_mel"].shape,
      "| MFCC vectors:", data["X_vec"].shape,
      "| labels:", data["y"].shape)
for name in ["train", "val", "test"]:
    print(f"  {name:5s}: {(data['split'] == name).sum()} clips")

## Spectral signature of each emotion

Averaging the log-Mel spectrograms within each class reveals a clear acoustic structure:
high-arousal emotions concentrate energy in the higher Mel bands, whereas low-arousal emotions are
quieter and dominated by low frequencies. This is direct visual evidence that the classes are at
least partially separable in the time–frequency plane.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for i, emo in enumerate(config.EMOTIONS):
    ax = axes[i // 4, i % 4]
    mask = (data["y"] == config.EMOTION_TO_ID[emo])
    librosa.display.specshow(data["X_mel"][mask].mean(0), sr=config.SAMPLE_RATE,
                             hop_length=config.HOP_LENGTH, x_axis="time", y_axis="mel", ax=ax)
    ax.set_title(f"{emo}  (n={mask.sum()})"); ax.set_xlabel("Time (s)"); ax.label_outer()
plt.suptitle("Class-averaged log-Mel spectrograms (RAVDESS)", fontsize=14)
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "01_mean_spectrograms.png", dpi=150)
plt.show()